In [ ]:
# Cell 1 — Đọc raw, không làm gì cả
import pm4py
import pandas as pd

log = pm4py.read_xes('../data/raw/BPI_2017.xes')
df_raw = pm4py.convert_to_dataframe(log)

print(f"Tổng số event: {len(df_raw)}")
print(f"Tổng số cột: {len(df_raw.columns)}")
print(f"\nDanh sách tất cả cột:")
for col in df_raw.columns:
    print(f"  {col}")

In [ ]:
# Cell 2 — Xem kiểu dữ liệu và tỉ lệ null từng cột
print("Kiểu dữ liệu và % null:\n")
for col in df_raw.columns:
    null_pct = df_raw[col].isna().mean() * 100
    dtype    = df_raw[col].dtype
    print(f"  {col:<45} {str(dtype):<15} null: {null_pct:.1f}%")

In [ ]:
# Cell 3 — Xem 5 dòng đầu để hiểu format thực tế
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
df_raw.head(5)

In [ ]:
# Cell 4 — Khảo sát lifecycle (rất quan trọng)
if 'lifecycle:transition' in df_raw.columns:
    print("Các giá trị lifecycle:transition:")
    print(df_raw['lifecycle:transition'].value_counts())
    print(f"\nTổng event START:    {(df_raw['lifecycle:transition'].str.upper()=='START').sum()}")
    print(f"Tổng event COMPLETE: {(df_raw['lifecycle:transition'].str.upper()=='COMPLETE').sum()}")

In [ ]:
# Cell 5 — Khảo sát case-level attributes
case_cols = [c for c in df_raw.columns if c.startswith('case:')]
print(f"Có {len(case_cols)} case-level attributes:\n")
for col in case_cols:
    unique_vals = df_raw[col].nunique()
    sample_vals = df_raw[col].dropna().unique()[:5]
    print(f"  {col}")
    print(f"    unique: {unique_vals}, sample: {sample_vals}\n")

In [ ]:
# Cell 6 — Khảo sát event-level attributes
event_cols = [c for c in df_raw.columns if not c.startswith('case:')]
print(f"Có {len(event_cols)} event-level attributes:\n")
for col in event_cols:
    unique_vals = df_raw[col].nunique()
    sample_vals = df_raw[col].dropna().unique()[:5]
    print(f"  {col}")
    print(f"    unique: {unique_vals}, sample: {sample_vals}\n")

In [ ]:
# Cell 7 — Xem 1 case cụ thể để hiểu cấu trúc thực tế
first_case = df_raw['case:concept:name'].iloc[0]
case_detail = df_raw[df_raw['case:concept:name'] == first_case].sort_values('time:timestamp')
print(f"Chi tiết case: {first_case}")
print(f"Số event: {len(case_detail)}\n")
case_detail[['time:timestamp', 'concept:name', 'lifecycle:transition', 'org:resource']].to_string(index=False)

In [2]:
import json
import pandas as pd

# 1. Số sequence (Path đúng: data/raw/sop/regulation_graph.json)
with open('../data/raw/sop/regulation_graph.json') as f:
    reg = json.load(f)
print('Activities :', len(reg['activities']))
print('Sequences  :', len(reg['sequences']))    # đáp án: 7
print('Conditions :', len(reg['conditions']))
print('Roles      :', len(reg['roles']))

# 2. Phạm vi thời gian dataset
df = pd.read_parquet('../data/processed/event_log_clean.parquet')
print('\nThời gian:')
print('  Min:', df['timestamp'].min())
print('  Max:', df['timestamp'].max())
print('  Số case:', df['case_id'].nunique())
print('  Số event:', len(df))

# 3. Số case happy-path (đáp án: 24895, không phải 18504)
df_app = df[df['event_origin'] == 'Application'].copy()
df_app = df_app.sort_values(['case_id', 'seq_index'])
end_events = df_app.groupby('case_id').last()
happy_set = {'A_Pending', 'A_Cancelled', 'A_Complete'}
happy_cases = end_events[end_events['activity'].isin(happy_set)].index
print(f'\nHappy-path cases: {len(happy_cases)}')

# 4. Số quan hệ trong RKG (cần Neo4j)
import os, sys
sys.path.append('../')
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv('../.env')
driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"),
          os.getenv("NEO4J_PASSWORD"))
)

with driver.session() as session:
    queries = [
        ('MUST_PRECEDE',  "MATCH ()-[r:MUST_PRECEDE]->() RETURN count(r) AS cnt"),
        ('REQUIRES',      "MATCH ()-[r:REQUIRES]->() RETURN count(r) AS cnt"),
        ('PERFORMED_BY',  "MATCH ()-[r:PERFORMED_BY]->() RETURN count(r) AS cnt"),
        ('DEFINED_IN',    "MATCH ()-[r:DEFINED_IN]->() RETURN count(r) AS cnt"),
    ]
    print('\nQuan hệ trong Neo4j:')
    for name, q in queries:
        result = session.run(q).single()
        print(f'  {name:<14}: {result["cnt"]}')

driver.close()

Activities : 13
Sequences  : 6
Conditions : 4
Roles      : 3

Thời gian:
  Min: 2016-01-01 09:51:15.304000+00:00
  Max: 2017-02-01 14:11:03.499000+00:00
  Số case: 28512
  Số event: 1047482

Happy-path cases: 24895

Quan hệ trong Neo4j:
  MUST_PRECEDE  : 6
  REQUIRES      : 0
  PERFORMED_BY  : 13
  DEFINED_IN    : 17
